# Countdown RFT from the SFT start with the label-free judge (Qwen)

Rejection-sampling fine-tuning (RFT / expert iteration), Consistency_RFT_Countdown34's method (the Llama run rft1-cut20),
with everything else as the Qwen GRPO runs (Consistency_GRPO_Qwen_Judge_Countdown): S's adapter as the start, part 1's
corrupt judge with the GRPO judge arm's reward (format + calibrated judge sum x3, missing at the 5th percentile), the
training puzzle file (rounds 1-4 = the GRPO runs' 1,200 puzzles), the held-out curve and S's grade puzzles.

Each round ([rft]): sample `RFT_PUZZLES` training puzzles x 16 at T 0.9 with the current policy, score each completion
with that reward (`SELECT = "checker"`: format + checker, the supervised baseline), keep per puzzle the top `RFT_KEEP`
finished completions above the group mean minus the lowest `RFT_SKIP_Q` of the round's picks, SFT on them, held-out
curve. 8 rounds x 300 x 16 = 38,400 rollouts (round 4 = the GRPO runs' 19,200). Label-free: the checker only measures.

Run order: Run all ([install] needs no restart). [rft] resumes after the last finished round (adapter + optimizer in
`RUN_DIR/round<N>/`); grades and the S curve are skipped when their files exist.

In [ ]:
# [config] Everything you choose lives here.
MODELS = {   # key: HF id, chat-template kwargs (Qwen3: thinking off, as the SFT and the screens)
    "qwen3-0.6b": ("Qwen/Qwen3-0.6B", {"enable_thinking": False}),
    "qwen3-1.7b": ("Qwen/Qwen3-1.7B", {"enable_thinking": False}),
}
MODEL_KEY = "qwen3-0.6b"
MODEL, CHAT_KWARGS = MODELS[MODEL_KEY]
SEED = 0                         # puzzle draws and eval sampling seeds: keep 0 (the grade files are draw 0)
SMOKE = False                    # True: every cell end to end on a few puzzles, 2 rounds of 8 puzzles (-smoke)
# S: Consistency_SFT_Countdown's run (same settings there)
SFT_RUN = f"{MODEL.split('/')[-1]}-countdown500-ep1-lr0.0001-seed0-tr4-f16-8-w35-30-25-10-imp0x0-mix3x500"
TEMPERATURE = 0.9
MAX_COMPLETION_LENGTH = 1024     # cap, as the Qwen GRPO runs and the Llama runs
# The judge: part 1's (Consistency_Qwen_Judge_Countdown) DPO adapter, trained on "empty" corruptions
JUDGE_RUN = ("judge_corrupt_empty-cut0.1-cut0.25-cut0.5_unfinished_n4x1000_n3x1000_t0.9_cap2048_seed0/"
             "dpo_empty_b0.1_lr2e-05_ep2")
N_CORRUPT_4, N_CORRUPT_3 = 1000, 1000            # part 1's corruption puzzles: kept out of the training puzzles
MAX_SOLUTION_TOKENS = 128        # the judge scores solution + answer of at most this many tokens; else missing
JUDGE_REDUCTION = "sum"
# MISSING_AT / JUDGE_WEIGHT_SCALE: the Qwen GRPO judge run's reward (x3, missing at the 5th percentile, -63.7)
MISSING_AT = "p5"
JUDGE_WEIGHT_SCALE = 3.0
EXPRESSION_REWARD = True

# ---- RFT (Consistency_RFT_Countdown34's, the Llama run rft1-cut20). Each of RFT_ROUNDS rounds takes the next
# RFT_PUZZLES training puzzles x NUM_GENERATIONS samples at TEMPERATURE with the current policy, scores each completion
# with the GRPO reward of the selection signal:
#   SELECT "judge"    format rewards + the calibrated judge (the GRPO judge arm's reward): label-free
#   SELECT "checker"  format rewards + the checker's correctness (2.0 if right): the supervised RFT baseline
# and per puzzle keeps the top RFT_KEEP FINISHED completions whose score is above the group mean (GRPO's positive
# advantages); RFT_SKIP_Q then drops the selected ones scored below that quantile of the round's selected scores.
# Then SFT on them: RFT_EPOCHS epoch(s), loss on the completion tokens (per-token mean over the batch), policy LoRA
# only, AdamW at RFT_LR (constant), batches of RFT_BATCH (micro-batches of RFT_MICRO), grad clip RFT_MAX_GRAD_NORM.
# No KL term. 8 x 300 x 16 = 38,400 rollouts; round 4 = 19,200 = the GRPO runs' 150 steps x 128.
SELECT = "judge"
RFT_ROUNDS, RFT_PUZZLES, RFT_KEEP = 8, 300, 2
RFT_LR, RFT_EPOCHS, RFT_BATCH, RFT_MICRO, RFT_MAX_GRAD_NORM = 2e-5, 1, 16, 4, 1.0
RFT_SKIP_Q = 0.2
NUM_GENERATIONS = 16
GRADE_ROUNDS = (4, 8)            # graded on S's grade puzzles (round 4 = the GRPO runs' rollout budget)

# [build]'s GRPO trainer, as the Qwen GRPO notebook: here it only holds the model (policy "default" = S, TRL's frozen
# "ref" = S) and the vLLM engine; nothing calls trainer.train(). ADAM_BETA2 and WEIGHT_DECAY are also RFT's AdamW.
PUZZLES_PER_STEP = 8             # the training puzzle file's layout: 4 four- + 4 three-number puzzles per block
MAX_STEPS = RFT_ROUNDS * RFT_PUZZLES // PUZZLES_PER_STEP     # blocks in the puzzle file (300; the first 150 = GRPO's)
LEARNING_RATE, WARMUP_STEPS = 2e-5, 20
KL_BETA = 0.02
LOSS_TYPE, SCALE_REWARDS, MAX_GRAD_NORM = "dapo", False, 0.2
VLLM_IS_MODE, VLLM_IS_CLIP = "token_truncate", 2.0
ADAM_BETA2, WEIGHT_DECAY = 0.99, 0.0
MICRO_BATCH = 8
SAVE_STEPS, DUMP_EVERY = 25, 25
VLLM_GPU_MEMORY_UTILIZATION = 0.5
CURVE_N, CURVE_K = 100, 4        # held-out curve after every round: CURVE_N per size, greedy x 1 + sampled x CURVE_K
N_GRADE, GRADE_SAMPLES = 200, 8                  # S's grade puzzles per size, as S's grade file
UNASSIGN_WHEN_DONE = True                        # [release]
if SMOKE:
    RFT_ROUNDS, RFT_PUZZLES, GRADE_ROUNDS = 2, 8, (2,)
    MAX_STEPS = RFT_ROUNDS * RFT_PUZZLES // PUZZLES_PER_STEP
    CURVE_N, CURVE_K, N_GRADE = 4, 2, 4

DRIVE = "/content/drive/MyDrive/Consistency"
CODE_DIR = f"{DRIVE}/code/countdown"
_sm = "-smoke" if SMOKE else ""
S_DIR = f"{DRIVE}/countdown/sft/{SFT_RUN}"
S_ADAPTER = f"{S_DIR}/adapter"
S_GRADE = f"{S_DIR}/grade_n200x8_t{TEMPERATURE:g}_cap{MAX_COMPLETION_LENGTH}_seed{SEED}{_sm}/gen_sft.csv"   # [judge] makes it if missing
GRADE_FILES = {n: f"{DRIVE}/countdown/grpo/olmo3-7b-format50-cap1024-seed0/grade/gen_Olmo-3-7B-Instruct-format50_user_n{n}.csv"
               for n in (4, 3)}          # the grade puzzles (S's were drawn from these)
JUDGE_DIR = f"{S_DIR}/{JUDGE_RUN}"
JUDGE_PATH = f"{JUDGE_DIR}/adapter"
CALIB_PATH = f"{JUDGE_DIR}/calibration_{JUDGE_REDUCTION}{ {'wrong_median': '_missmed', 'p5': ''}[MISSING_AT]}_S_{S_GRADE.split('/')[-2]}{_sm}.json"   # made by [judge]
# The run folder is {model}-sft-rft-{SELECT}-{RUN_TAG}; the settings live in its run.json. Rerunning a folder with other
# settings asks first ([rft]).
RUN_TAG = "w3-p5-cut20"
RUN_ID = f"{MODEL_KEY}-sft-rft-{SELECT}-{RUN_TAG}{_sm}"
RUN_DIR = f"{DRIVE}/countdown/grpo/{RUN_ID}"
CURVE_S = f"{S_DIR}/curve_S_n34x{CURVE_N}_greedy_pass{CURVE_N}x{CURVE_K}_t{TEMPERATURE:g}_cap{MAX_COMPLETION_LENGTH}_seed{SEED}{_sm}"
# Training puzzles: the GRPO runs' file (read only; its 150 blocks = rounds 1-4), grown by the same rule to MAX_STEPS
# blocks and saved as this notebook's own file
TRAIN_PUZZLES_SRC = f"{S_DIR}/grpo_train_puzzles_n34_x{PUZZLES_PER_STEP}_seed{SEED}{_sm}.csv"
TRAIN_PUZZLES_CSV = f"{S_DIR}/rft_train_puzzles_n34_x{PUZZLES_PER_STEP}_steps{MAX_STEPS}_seed{SEED}{_sm}.csv"
# Graded Qwen GRPO runs (same S, judge, grade puzzles, cap) for [compare]: label -> run id (final adapter, 150 steps)
COMPARE_RUNS = {"GRPO judge x3": "qwen3-0.6b-sft-judge-sum-empty-w3-n34-steps150-cap1024-istt2-seed0",
                "GRPO format": "qwen3-0.6b-sft-format-n34-steps150-cap1024-istt2-seed0"}
assert SELECT in ("judge", "checker") and JUDGE_REDUCTION in ("mean", "sum")
assert MISSING_AT in ("p5", "wrong_median"), MISSING_AT
assert (RFT_ROUNDS * RFT_PUZZLES) % PUZZLES_PER_STEP == 0 and PUZZLES_PER_STEP % 2 == 0
assert all(1 <= r <= RFT_ROUNDS for r in GRADE_ROUNDS), GRADE_ROUNDS
print(f"MODEL={MODEL} {CHAT_KWARGS}\nS={S_ADAPTER}\nJUDGE_DIR={JUDGE_DIR}\nRUN_DIR={RUN_DIR}")

In [ ]:
# [drive + HF] Mount Drive, HF login (Colab secret HF_TOKEN), and the checks that must fail before any GPU work.
import os, glob, shutil, json
from google.colab import drive, userdata
if not os.path.ismount("/content/drive"):
    drive.mount("/content/drive")
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
for f in ("countdown.py", "prompts.py", "rewards.py"):
    assert os.path.exists(f"{CODE_DIR}/{f}"), f"missing {CODE_DIR}/{f}"
assert os.path.exists(f"{S_ADAPTER}/adapter_model.safetensors"), f"no S adapter in {S_ADAPTER}"
for f in GRADE_FILES.values():
    assert os.path.exists(f), f"missing {f}"
assert os.path.exists(f"{JUDGE_PATH}/adapter_model.safetensors"), f"no judge in {JUDGE_PATH}: run part 1 first"
os.makedirs(RUN_DIR, exist_ok=True)
print("judge:", JUDGE_PATH, "| calibration:", "exists" if os.path.exists(CALIB_PATH) else "made by [judge]")
print("training puzzles:", TRAIN_PUZZLES_CSV if os.path.exists(TRAIN_PUZZLES_CSV) else
      f"grown from {TRAIN_PUZZLES_SRC}" if os.path.exists(TRAIN_PUZZLES_SRC) else "drawn ([puzzles])")
_rd_done = [r for r in range(1, RFT_ROUNDS + 1) if os.path.exists(f"{RUN_DIR}/round{r}/optimizer.pt")]
print(f"{RUN_ID}: " + (f"rounds 1-{max(_rd_done)} done" if _rd_done else "from S"))

In [ ]:
# [install] Once per fresh runtime, no restart needed (Consistency_GRPO_Olmo_Judge's): skipped when TRL >= 1 and vLLM are
# installed. pip runs as a subprocess, not %pip, so Colab shows no "restart the session" prompt; Pillow is pinned to the
# version Colab already imported. torchao is removed (peft refuses Colab's 0.10).
import importlib.metadata as _md, importlib.util, subprocess, sys
try:
    _ready = int(_md.version("trl").split(".")[0]) >= 1 and importlib.util.find_spec("vllm") is not None
except _md.PackageNotFoundError:
    _ready = False
if _ready:
    print(f"[install] skipped: trl {_md.version('trl')} and vllm {_md.version('vllm')} are installed")
else:
    _pin = [f"pillow=={_md.version('pillow')}"] if importlib.util.find_spec("PIL") else []
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "trl[vllm]", "peft", *_pin], check=True)
    print(f"[install] done: trl {_md.version('trl')}, vllm {_md.version('vllm')}" + (f", {_pin[0]} kept" if _pin else ""))
if importlib.util.find_spec("torchao"):
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=True)
    print("[install] removed torchao")

In [ ]:
# [vLLM setup] Before anything touches the GPU. (No expandable_segments: with it vLLM's CUDA-graph capture failed,
# Consistency_MATH_Finish_Probe, 2026-11-12.)
import os, sys, subprocess
os.environ["VLLM_ENABLE_V1_MULTIPROCESSING"] = "0"  # keep the vLLM engine in this process
os.environ["TRL_EXPERIMENTAL_SILENCE"] = "1"         # rollout_func is experimental in TRL
for stream, fd in ((sys.stdout, 1), (sys.stderr, 2)):   # Colab's stdout/stderr have no OS file descriptor
    try:
        stream.fileno()
    except Exception:
        stream.fileno = lambda fd=fd: fd
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,memory.used", "--format=csv"],
                     capture_output=True, text=True).stdout)
import importlib.metadata as md
for p in ["torch", "transformers", "trl", "peft", "datasets", "vllm"]:
    try:
        print(f"{p:14s}", md.version(p))
    except md.PackageNotFoundError:
        print(f"{p:14s}", "not installed")

In [ ]:
# [setup] Tokenizer (CHAT_KWARGS on every template call), the Countdown code (Drive), the labels (the screen's / S's
# grade file's), the answer classes (corrupt-judge notebooks), and the judge's text views.
import re, json, random, importlib, math, gc, time
from collections import Counter
from fractions import Fraction
import numpy as np
import pandas as pd
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained(MODEL)
assert tok.pad_token is not None and tok.pad_token_id != tok.eos_token_id, tok.pad_token

def chat_text(messages):
    return tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, **CHAT_KWARGS)

def chat_ids(messages):
    return tok(chat_text(messages), add_special_tokens=False)["input_ids"]

sys.path.insert(0, CODE_DIR)
import countdown, prompts, rewards as R
for _m in (countdown, prompts, R):
    importlib.reload(_m)
from countdown import check_answer
from prompts import countdown_prompt, judge_messages
tag, completion_text = R.tag, R.completion_text
_b = lambda d, c: d[c].astype(str) == "True"
_rd = lambda p: pd.read_csv(p, keep_default_na=False, dtype={c: str for c in ("completion", "reasoning", "solution", "answer",
                                                                              "generated", "text")})

def label(df):
    """As Consistency_SFT_Countdown's [grade] (the screen's): blocks, format_ok, answered, steps_parsed, correct, diagnosis."""
    df = df.copy()
    texts, nums, targets = list(df.completion), [json.loads(x) for x in df.numbers], list(df.target)
    for b in ("reasoning", "solution", "answer"):
        df[b] = [R.tag(t, b) or "" for t in texts]
    df["format_ok"] = [bool(re.search(r"<reasoning>.*?</reasoning>\s*<solution>.*?</solution>\s*<answer>.*?</answer>",
                                      t, re.DOTALL)) for t in texts]
    df["answered"] = df.answer.str.strip() != ""
    df["steps_parsed"] = [bool(R.parse_steps(s)) for s in df.solution]
    df["correct"] = np.array(R.correctness_reward_func(None, texts, nums, targets)) > 0
    df["diagnosis"] = [R.diagnose(a or None, n, t, tr) for a, n, t, tr in zip(df.answer, nums, targets, df.truncated)]
    return df

def read_graded(path):
    d = _rd(path)
    for c in ("finished", "truncated", "format_ok", "answered", "steps_parsed", "correct"):
        d[c] = _b(d, c)
    d["puzzle"], d["n_numbers"] = d.puzzle.astype(int), d.n_numbers.astype(int)
    return d

def stats(d):
    """As the SFT grade's table (cap MAX_COMPLETION_LENGTH)."""
    fin = d.finished
    ans, ok = fin & d.answered, fin & d.correct
    g = d.assign(aw=ans & ~ok, ok=ok).groupby("puzzle")
    return {"n": len(d), "accuracy": ok.mean(), "format ok": (fin & d.format_ok).mean(),
            "solution parses": (fin & d.steps_parsed).mean(), "truncated": 1 - fin.mean(),
            "answered wrong": (ans & ~ok).mean(), "correct among answered": ok.sum() / max(ans.sum(), 1),
            "tokens (median, finished)": d.n_tokens[fin].median(),
            "puzzles: correct + answered wrong": int((g.ok.any() & g.aw.any()).sum()), "pass@k": g.ok.any().mean()}

_NUM = r"\(?\s*(-?\d+(?:\.\d+)?(?:\s*/\s*\d+)?)\s*\)?"
_STEP = re.compile(rf"^\s*{_NUM}\s*([+\-*/×÷x])\s*{_NUM}\s*=\s*{_NUM}\s*\.?\s*$")

def replay(sol, numbers):
    """"false" / "unavailable" / ("valid", last result) / "unparsed" (the corrupt-judge notebooks')."""
    avail = Counter(Fraction(x) for x in numbers)
    bad = False
    lines = [l for l in (sol or "").splitlines() if l.strip()]
    if not lines:
        return "unparsed"
    for line in lines:
        m = _STEP.match(line.split("Left:")[0])
        if not m:
            return "unparsed"
        a, op, b, c = m.groups()
        a, b, c = (Fraction(s.replace(" ", "")) for s in (a, b, c))
        r = {"+": a + b, "-": a - b, "*": a * b, "×": a * b, "x": a * b}.get(op, a / b if b else None)
        if r != c:
            return "false"
        for v in (a, b):
            bad |= avail[v] < 1
            avail[v] -= 1
        avail[c] += 1
    return "unavailable" if bad else ("valid", c)

def klass(sol, ans, numbers, target):
    if not isinstance(ans, str) or not ans.strip():
        return "no answer"
    if check_answer(ans, list(numbers), int(target)):
        return "correct"
    r = replay(sol, numbers)
    if isinstance(r, tuple):
        return "bad answer" if r[1] == target else "honest miss"
    return {"false": "fake", "unavailable": "wrong numbers", "unparsed": "other"}[r]

def scorable(sol, ans):
    """Scored by the GRPO reward: a solution of at most MAX_SOLUTION_TOKENS tokens and an answer."""
    return (isinstance(sol, str) and bool(sol.strip()) and isinstance(ans, str) and bool(ans.strip())
            and len(tok(sol, add_special_tokens=False)["input_ids"]) <= MAX_SOLUTION_TOKENS)

judge_text = lambda sol, ans: f"{sol.strip()}\nAnswer: {ans.strip()}"

def unfinished_text(generated):
    """What was written inside <solution> without reaching an answer: up to </solution>, or all of it."""
    return generated.split("</solution>", 1)[0].strip()

def end_of_turn_id(tok):
    s = tok.apply_chat_template([{"role": "user", "content": "Q"}, {"role": "assistant", "content": "XYZXYZ"}],
                                tokenize=False, **CHAT_KWARGS)
    eot = tok(s.split("XYZXYZ", 1)[1], add_special_tokens=False)["input_ids"][0]
    assert tok.convert_ids_to_tokens(eot) in tok.all_special_tokens, tok.convert_ids_to_tokens(eot)
    return eot

EOT = end_of_turn_id(tok)
_p = chat_text([{"role": "user", "content": "x"}])
if CHAT_KWARGS.get("enable_thinking") is False:
    assert _p.rstrip().endswith("</think>"), "thinking-off template should end with an empty think block"
print("tokenizer:", MODEL, "| pad", tok.pad_token, "| eos", tok.eos_token, "| end of turn", tok.decode([EOT]),
      "| generation prompt ends", repr(_p[-40:]))

In [ ]:
# [puzzles] From Jiayi-Pan/Countdown-Tasks-3to4, disjoint by construction:
#   grade       S's grade puzzles (the grade files: 200 four- + 200 three-number of the test draw SEED), first N_GRADE
#   corruption  part 1's (draw SEED + 6), only kept out of the training and curve puzzles
#   training    TRAIN_PUZZLES_CSV, in blocks of PUZZLES_PER_STEP / 2 four- then as many three-number puzzles: the GRPO
#               runs' file (TRAIN_PUZZLES_SRC, read only) grown to MAX_STEPS blocks by the rule below, saved as this
#               notebook's own file. (GRPO notebook's rule:)
#               New file: draw SEED + 8, MAX_STEPS steps, apart from all of the above (whole draws) and the corruption
#               draw. A file with fewer steps than MAX_STEPS keeps its steps and grows by draw SEED + 12 (apart from
#               everything, the curve too); one with more is used up to MAX_STEPS.
#   curve       draw SEED + 10: CURVE_N of each size, apart from everything above (the file's first training steps)
_tz = load_dataset("Jiayi-Pan/Countdown-Tasks-3to4", split="train")
_as = lambda r: {"numbers": [int(x) for x in r["nums"]], "target": int(r["target"])}
_key = lambda p: (tuple(sorted(p["numbers"])), p["target"])
_seeded = lambda seed, size: _tz.select(np.random.default_rng(seed).choice(len(_tz), size=min(len(_tz), size), replace=False))

GRADE = {}
for n, f in GRADE_FILES.items():
    g = pd.read_csv(f, keep_default_na=False).drop_duplicates("puzzle").sort_values("puzzle").head(N_GRADE)
    GRADE[n] = [{"puzzle": int(r.puzzle), "numbers": json.loads(r.numbers), "target": int(r.target)} for r in g.itertuples()]
_draw0 = _seeded(SEED, 20 * 200)
assert [_key(p) for p in GRADE[4]] == [_key(_as(r)) for r in _draw0 if len(r["nums"]) == 4][1:N_GRADE + 1]

_no = {_key(_as(r)) for seed, size in ((SEED, 20 * 200), (SEED + 1, 6 * (4000 + 100)), (SEED + 2, 4 * (4000 + 100)),
                                        (SEED + 3, 4 * 3900)) for r in _seeded(seed, size)}
_want, _got = {4: N_CORRUPT_4, 3: N_CORRUPT_3}, {4: [], 3: []}
for r in _seeded(SEED + 6, 10 * (N_CORRUPT_4 + N_CORRUPT_3)):
    p = _as(r)
    n = len(p["numbers"])
    if _key(p) not in _no and len(_got[n]) < _want[n]:
        _no.add(_key(p))
        _got[n].append(p)
assert all(len(_got[n]) == _want[n] for n in _want), {n: len(v) for n, v in _got.items()}
corrupt_puzzles = _got[4] + _got[3]
_no |= {_key(_as(r)) for r in _seeded(SEED + 6, 10 * (N_CORRUPT_4 + N_CORRUPT_3))}

_half = PUZZLES_PER_STEP // 2
_from_df = lambda df: [{"numbers": json.loads(r.numbers), "target": int(r.target)} for r in df.itertuples()]
_src = TRAIN_PUZZLES_CSV if os.path.exists(TRAIN_PUZZLES_CSV) else TRAIN_PUZZLES_SRC   # the GRPO file: read only
if os.path.exists(_src):
    _train = _from_df(pd.read_csv(_src))
    assert len(_train) % PUZZLES_PER_STEP == 0 and all(
        len(p["numbers"]) == (4 if i % PUZZLES_PER_STEP < _half else 3) for i, p in enumerate(_train)), _src
else:
    _need = MAX_STEPS * _half
    _tr = {4: [], 3: []}
    for r in _seeded(SEED + 8, 40 * _need):
        p = _as(r)
        n = len(p["numbers"])
        if _key(p) not in _no and len(_tr[n]) < _need:
            _no.add(_key(p))
            _tr[n].append(p)
    assert all(len(v) == _need for v in _tr.values()), {n: len(v) for n, v in _tr.items()}
    _train = [p for s in range(MAX_STEPS) for n in (4, 3) for p in _tr[n][s * _half:(s + 1) * _half]]
_saved_steps = len(_train) // PUZZLES_PER_STEP
_write = _saved_steps < MAX_STEPS or not os.path.exists(TRAIN_PUZZLES_CSV)
_no |= {_key(p) for p in _train}
CURVE = {4: [], 3: []}
for r in _seeded(SEED + 10, 40 * CURVE_N):
    p = _as(r)
    n = len(p["numbers"])
    if _key(p) not in _no and len(CURVE[n]) < CURVE_N:
        _no.add(_key(p))
        CURVE[n].append(p)
assert all(len(v) == CURVE_N for v in CURVE.values())
if _saved_steps < MAX_STEPS:                     # grow the file: the extra steps only
    _need = (MAX_STEPS - _saved_steps) * _half
    _tr = {4: [], 3: []}
    for r in _seeded(SEED + 12, 40 * _need):
        p = _as(r)
        n = len(p["numbers"])
        if _key(p) not in _no and len(_tr[n]) < _need:
            _no.add(_key(p))
            _tr[n].append(p)
    assert all(len(v) == _need for v in _tr.values()), {n: len(v) for n, v in _tr.items()}
    _train += [p for s in range(MAX_STEPS - _saved_steps) for n in (4, 3) for p in _tr[n][s * _half:(s + 1) * _half]]
    print(f"[puzzles] training file: {_saved_steps} steps kept, {MAX_STEPS - _saved_steps} drawn")
_train = _train[:MAX_STEPS * PUZZLES_PER_STEP]
_tdf = pd.DataFrame([{"pid": i, "n": len(p["numbers"]), "numbers": json.dumps(p["numbers"]), "target": p["target"]}
                     for i, p in enumerate(_train)])
if _write:
    _tdf.to_csv(TRAIN_PUZZLES_CSV, index=False)
train_dataset = Dataset.from_list([{"prompt": countdown_prompt(p["numbers"], p["target"]), "numbers": p["numbers"],
                                    "target": p["target"], "pid": i, "n": len(p["numbers"])} for i, p in enumerate(_train)])
_gk = {_key(p) for v in GRADE.values() for p in v}
assert not _gk & ({_key(p) for p in _train} | {_key(p) for p in corrupt_puzzles} | {_key(p) for v in CURVE.values() for p in v})
print(f"grade {({n: len(v) for n, v in GRADE.items()})}, corruption (excluded) {len(corrupt_puzzles)}, training {len(train_dataset)} ({MAX_STEPS} blocks x {_half} + {_half}), "
      f"curve {({n: len(v) for n, v in CURVE.items()})}")

In [ ]:
# [vLLM sync fix] (Consistency_GRPO_Olmo_Judge's) TRL syncs LoRA to vLLM by merging dW into the training model's bf16 W
# in place, then unmerging; each add/subtract rounds, so the frozen W drifts. This version streams bf16(W + dW) built in
# a temporary tensor (one rounding) and never touches the training model. Only active adapters are merged.
import torch
from accelerate.utils import is_peft_model
from peft.tuners.lora import LoraLayer
from trl.generation.vllm_generation import VLLMGeneration

if not hasattr(VLLMGeneration, "_trl_iter_named_params"):
    VLLMGeneration._trl_iter_named_params = VLLMGeneration._iter_named_params

def _iter_named_params_out_of_place(self):
    model = self.model
    if not is_peft_model(model) or self.accelerator.num_processes > 1:
        yield from self._trl_iter_named_params()
        return
    lora_layers = {name: m for name, m in model.named_modules() if isinstance(m, LoraLayer)}
    n_merged = 0
    for name, param in model.named_parameters():
        vllm_name = name.removeprefix("base_model.model.").replace(".base_layer", "")
        if model.prefix in vllm_name or "original_module" in vllm_name:
            continue
        vllm_name = self._fix_param_name_to_vllm(vllm_name, extra_prefixes=["modules_to_save.default."])
        layer = lora_layers.get(name.rsplit(".base_layer.", 1)[0]) if name.endswith(".base_layer.weight") else None
        if layer is None:
            yield vllm_name, param.data
            continue
        assert not layer.merged, f"{name}: adapter is merged in the training model"
        with torch.no_grad():
            merged = param.data.float()
            for adapter in layer.active_adapters:
                if adapter in layer.lora_A:
                    merged += layer.get_delta_weight(adapter).float()
            merged = merged.to(param.dtype)
        n_merged += 1
        yield vllm_name, merged
        del merged
    if not getattr(self, "_out_of_place_reported", False):
        print(f"[vLLM sync fix] streamed {n_merged} LoRA layers as bf16(W + dW); training weights untouched")
        self._out_of_place_reported = True

VLLMGeneration._iter_named_params = _iter_named_params_out_of_place
print("VLLMGeneration._iter_named_params patched")

In [ ]:
# [rewards] (the GRPO notebook's; [rft] uses format_total, judge_input and judge_scores: the GRPO judge arm's reward)
# S's format rewards (xmlcount, soft / strict format, expression); the judge (reward weight set per arm in
# [train]: calibrated in the judge arm, 0 in the format arm) scores, under its frozen adapter, the judge prompt +
# "solution\nAnswer: answer" + end of turn when both parse and the solution has <= MAX_SOLUTION_TOKENS tokens; else
# missing = JUDGE_MISSING (set by [judge]). Logged with weight 0: correctness, the judge's mean and sum, S's monitors;
# the step monitor prints every step and dumps every DUMP_EVERY to the arm's RUN_DIR.
import contextlib
from trl.trainer.utils import selective_log_softmax
CUR = {"arm": None, "run_dir": None}            # the arm [train] runs
JUDGE_MISSING = float("nan")                    # set by [judge]

def count_xml(text) -> float:
    count = 0.0
    if text.count("<reasoning>") == 1:
        count += 0.125
    if text.count("</reasoning>") == 1:
        count += 0.125
    if text.count("<solution>") == 1:
        count += 0.005
    if text.count("</solution>") == 1:
        count += 0.005
    if text.count("<answer>") == 1:
        count += 0.125
    if text.count("</answer>") == 1:
        count += 0.125
    if "</answer>" in text:
        count -= (len(text.split("</answer>", 1)[1]) - 1)*0.001
    return count

def xmlcount_reward_func(completions, **kwargs) -> list[float]:
    return [count_xml(completion_text(c)) for c in completions]

def soft_format_reward_func(completions, **kwargs) -> list[float]:
    pattern = r"<reasoning>[\s\S]*?</reasoning>\s*<solution>[\s\S]*?</solution>\s*<answer>[\s\S]*?</answer>"
    return [0.5 if re.match(pattern, completion_text(c)) else 0.0 for c in completions]

def strict_format_reward_func(completions, **kwargs) -> list[float]:
    pattern = r"<reasoning>\s*.*?\s*</reasoning>\s*<solution>\s*.*?\s*</solution>\s*<answer>\s*.*?\s*</answer>"
    return [0.5 if re.search(pattern, completion_text(c), re.DOTALL) else 0.0 for c in completions]

FORMAT_FUNCS = [xmlcount_reward_func, soft_format_reward_func, strict_format_reward_func] + (
    [R.expression_reward_func] if EXPRESSION_REWARD else [])

def format_total(texts):
    """The summed format reward of completion texts (what the format arm trains on)."""
    cs = [[{"role": "assistant", "content": t}] for t in texts]
    return np.sum([f(completions=cs) for f in FORMAT_FUNCS], axis=0)

def judge_input(text):
    """The text the judge scores, or None (missing)."""
    s, a = tag(text, "solution"), tag(text, "answer")
    return judge_text(s, a) if scorable(s, a) else None

def _freeze_others(model):
    for n, p in model.named_parameters():        # set_adapter makes the active adapter trainable: only the policy is
        if "lora_" in n and ".default." not in n:
            p.requires_grad_(False)

@contextlib.contextmanager
def scorer_weights(model, adapter):
    """adapter: "judge", "ref" (= S, frozen) or None (the untrained model: adapters off); then the policy again."""
    if adapter is None:
        with model.disable_adapter():
            yield
        return
    model.set_adapter(adapter)
    try:
        yield
    finally:
        model.set_adapter("default")
        _freeze_others(model)

def judge_scores(model, items, adapter="judge", chunk=16):
    """(mean, sum) arrays over items = [(numbers, target, text or None)], NaN for None: per-token log-probs of the text +
    end of turn after the judge prompt (Consistency_Corrupt_Judge_Olmo's score)."""
    means, sums = np.full(len(items), np.nan), np.full(len(items), np.nan)
    seqs = {i: (chat_ids(judge_messages(list(n), int(t))), tok(x, add_special_tokens=False)["input_ids"] + [EOT])
            for i, (n, t, x) in enumerate(items) if x}
    order = sorted(seqs, key=lambda i: len(seqs[i][0]) + len(seqs[i][1]))
    fwd = getattr(model, "_original_forward", model.forward)
    with torch.no_grad(), scorer_weights(model, adapter):
        for s in range(0, len(order), chunk):
            idx = order[s:s + chunk]
            L = max(len(seqs[i][0]) + len(seqs[i][1]) for i in idx)
            ids = torch.full((len(idx), L), tok.pad_token_id, dtype=torch.long)
            attn = torch.zeros_like(ids)
            for j, i in enumerate(idx):
                x = seqs[i][0] + seqs[i][1]
                ids[j, :len(x)], attn[j, :len(x)] = torch.tensor(x), 1
            ids, attn = ids.to(model.device), attn.to(model.device)
            per_pos = selective_log_softmax(fwd(input_ids=ids, attention_mask=attn).logits[:, :-1], ids[:, 1:]).float()
            for j, i in enumerate(idx):
                p, t = seqs[i]
                lp = per_pos[j, len(p) - 1:len(p) + len(t) - 1].cpu().numpy()
                means[i], sums[i] = lp.mean(), lp.sum()
            del per_pos
    return means, sums

_LAST = {}

def make_judge_reward(owner):
    def judge(completions, numbers, target, **kwargs):
        assert "judge" in owner.model.peft_config and not math.isnan(JUDGE_MISSING), "run [judge] first"
        items = [(n, t, judge_input(completion_text(c))) for n, t, c in zip(numbers, target, completions)]
        m, s = judge_scores(owner.model, items)
        _LAST.update(id=id(completions), mean=m, sum=s)
        v = m if JUDGE_REDUCTION == "mean" else s
        return [JUDGE_MISSING if math.isnan(x) else float(x) for x in v]
    judge.__name__ = "judge"
    return judge

def _last(key, completions):
    v = _LAST.get(key) if _LAST.get("id") == id(completions) else None
    return [None] * len(completions) if v is None else [None if math.isnan(x) else float(x) for x in v]

def judge_mean(completions, **kwargs):
    return _last("mean", completions)

def judge_sum(completions, **kwargs):
    return _last("sum", completions)

def correct(completions, numbers, target, **kwargs):
    return [float(x > 0) for x in R.correctness_reward_func(None, completions, numbers, target)]

MONITORS = list(R.MONITORS)

def within_puzzle_win_rate(keys, scores, correct):
    """Over (correct, wrong) pairs of the same puzzle: share where the correct one scores higher, ties 1/2."""
    wins = pairs = 0
    keys = np.array(keys)
    for k in set(keys.tolist()):
        s, c = scores[keys == k], correct[keys == k]
        d = s[c][:, None] - s[~c][None, :]
        wins += (d > 0).sum() + 0.5 * (d == 0).sum()
        pairs += d.size
    return (wins / pairs if pairs else float("nan")), int(pairs)

def make_step_monitor(owner):
    def step_monitor(prompts, completions, numbers, target, pid, **kwargs):
        step_monitor_check(completions)
        texts = [completion_text(c) for c in completions]
        cn = np.array(correct(completions, numbers, target)) > 0
        fin = np.array([("</answer>" in t) for t in texts])
        v = np.array(judge_mean(completions) if JUDGE_REDUCTION == "mean" else judge_sum(completions), dtype=float)
        v = np.where(np.isnan(v), JUDGE_MISSING, v)
        rate, pairs = within_puzzle_win_rate(pid, v, cn)
        n_ = np.array([len(x) for x in numbers])
        line = (f"correct {cn.mean():.3f} (4: {cn[n_ == 4].mean():.3f}, 3: {cn[n_ == 3].mean():.3f}) | answered "
                f"{fin.mean():.2f} | correct among answered "
                f"{cn[fin].mean() if fin.any() else float('nan'):.3f} | judge {JUDGE_REDUCTION} {v.mean():.3f}, "
                f"win {rate:.2f} ({pairs} pairs)")
        log = kwargs.get("log_metric")
        if log:
            log("correct_among_finished", float(cn[fin].mean()) if fin.any() else float("nan"))
            if pairs:
                log("judge_win_rate", float(rate))
        step = owner.state.global_step
        print(f"[{CUR['arm']} step {step}] {line}")
        if step % DUMP_EVERY == 0:
            with open(f"{CUR['run_dir']}/step_{step:05d}.txt", "w", encoding="utf-8") as f:
                f.write(f"=== Step {step} ===\n\n")
                for i, p in enumerate(prompts):
                    f.write(f"### Sample {i + 1}\n[Scores] puzzle={pid[i]} correct={bool(cn[i])} "
                            f"judge_{JUDGE_REDUCTION}={v[i]:.4f}\n"
                            f"[Prompt]\n{p[-1]['content'].strip()}\n\n[Response]\n{texts[i].strip()}\n\n"
                            + "=" * 80 + "\n\n")
        return [0.0] * len(completions)
    step_monitor.__name__ = "step_monitor"
    return step_monitor

def step_monitor_check(completions):
    """Qwen3 thinking off: no training completion may open <think> (TRL's chat_template_kwargs applied)."""
    if CHAT_KWARGS.get("enable_thinking") is False:
        _t = sum("<think>" in completion_text(c) for c in completions)
        assert _t == 0, f"{_t} training completions open <think>: TRL did not apply enable_thinking=False"
    return True

REWARD_FUNCS_TRAIN = ([(f, 1.0) for f in FORMAT_FUNCS] + [("judge", 0.0)]
                      + [(correct, 0.0), (judge_mean, 0.0), (judge_sum, 0.0)] + [(f, 0.0) for f in MONITORS] + [("step_monitor", 0.0)])
JUDGE_IDX = [f for f, _ in REWARD_FUNCS_TRAIN].index("judge")
print("reward functions:", [f if isinstance(f, str) else f.__name__ for f, _ in REWARD_FUNCS_TRAIN])

In [ ]:
# [build] (the GRPO notebook's; nothing calls trainer.train() here) The policy (S's adapter "default", trainable), the GRPO config and trainer (TRL copies S to a frozen "ref"
# adapter: the KL reference, and S's own scorer in [judge]); its vLLM engine samples everything in this notebook.
# Checks: adapters, the trainer's prompt tokens = chat_ids (thinking off), vLLM vs trainer log-probs. The judge adapter
# is added in [judge].
import csv, tempfile, inspect
from datetime import datetime
from transformers import AutoModelForCausalLM, TrainerCallback
from peft import PeftModel
from trl import GRPOTrainer, GRPOConfig
from vllm import SamplingParams

class CSVLoggerCallback(TrainerCallback):
    """Every log row to a CSV; the header grows when new keys appear."""
    def __init__(self, path):
        self.path, self.fields = path, None
        os.makedirs(os.path.dirname(path), exist_ok=True)
        if os.path.exists(path) and os.path.getsize(path):
            with open(path, newline="") as f:
                self.fields = next(csv.reader(f))
    def on_log(self, args, state, control, logs=None, **kwargs):
        if not logs:
            return
        logs = {"step": state.global_step, **logs}
        if self.fields is None:
            self.fields = list(logs)
            with open(self.path, "w", newline="") as f:
                csv.DictWriter(f, fieldnames=self.fields).writeheader()
        new = [k for k in logs if k not in self.fields]
        if new:
            self.fields += sorted(new)
            rows = list(csv.DictReader(open(self.path, newline="")))
            fd, tmp = tempfile.mkstemp(dir=os.path.dirname(self.path))
            os.close(fd)
            with open(tmp, "w", newline="") as f:
                w = csv.DictWriter(f, fieldnames=self.fields)
                w.writeheader()
                w.writerows(rows)
            shutil.move(tmp, self.path)
        with open(self.path, "a", newline="") as f:
            csv.DictWriter(f, fieldnames=self.fields).writerow(logs)

class JudgeGRPOTrainer(GRPOTrainer):
    def log(self, logs, start_time=None):
        super().log(logs, start_time)
        if "reward" in logs and "train_runtime" not in logs:
            v = lambda k: float("nan") if logs.get(k) is None else float(logs[k])
            print(f"[{CUR['arm']} {self.state.global_step}] reward={v('reward'):.3f}, no-gradient groups="
                  f"{v('frac_reward_zero_std'):.2f}, grad_norm={v('grad_norm'):.3f}, kl={v('kl'):.4f}, "
                  f"length={v('completions/mean_length'):.0f}, truncated={v('completions/clipped_ratio'):.3f}, "
                  f"correct={v('rewards/correct/mean'):.3f}, judge {JUDGE_REDUCTION}="
                  f"{v(f'rewards/judge_{JUDGE_REDUCTION}/mean'):.3f}")

model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.bfloat16, device_map="auto")
model.config.pad_token_id = tok.pad_token_id
model.generation_config.pad_token_id = tok.pad_token_id
model = PeftModel.from_pretrained(model, S_ADAPTER, is_trainable=True)          # "default" = S, continued per arm

COMPLETIONS_PER_STEP = PUZZLES_PER_STEP * NUM_GENERATIONS
training_args = GRPOConfig(
    use_vllm=True, vllm_mode="colocate", vllm_gpu_memory_utilization=VLLM_GPU_MEMORY_UTILIZATION,
    vllm_max_model_length=MAX_COMPLETION_LENGTH + 512,
    chat_template_kwargs=CHAT_KWARGS, gradient_checkpointing=True,
    loss_type=LOSS_TYPE, scale_rewards=SCALE_REWARDS, max_grad_norm=MAX_GRAD_NORM, temperature=TEMPERATURE,
    beta=KL_BETA, reward_weights=[w for _, w in REWARD_FUNCS_TRAIN],
    learning_rate=LEARNING_RATE, warmup_steps=WARMUP_STEPS, lr_scheduler_type="constant_with_warmup", optim="adamw_torch",
    adam_beta1=0.9, adam_beta2=ADAM_BETA2, weight_decay=WEIGHT_DECAY,
    vllm_importance_sampling_correction=True, vllm_importance_sampling_mode=VLLM_IS_MODE,
    vllm_importance_sampling_clip_max=VLLM_IS_CLIP,
    seed=SEED, shuffle_dataset=False, logging_strategy="steps", logging_steps=5,
    save_strategy="steps", save_steps=SAVE_STEPS, save_total_limit=None, output_dir=RUN_DIR, report_to=[],
    bf16=True, per_device_train_batch_size=MICRO_BATCH, gradient_accumulation_steps=COMPLETIONS_PER_STEP // MICRO_BATCH,
    num_generations=NUM_GENERATIONS, max_completion_length=MAX_COMPLETION_LENGTH, max_steps=MAX_STEPS,
)
_bound = {}

def _late(name):
    def f(**kwargs):
        return _bound[name](**kwargs)
    f.__name__ = name
    return f

trainer = JudgeGRPOTrainer(model=model, processing_class=tok, args=training_args, train_dataset=train_dataset,
                           reward_funcs=[_late(f) if isinstance(f, str) else f for f, _ in REWARD_FUNCS_TRAIN])
_bound.update(judge=make_judge_reward(trainer), step_monitor=make_step_monitor(trainer))
_pm = trainer.model
assert {"default", "ref"} <= set(_pm.peft_config), f"adapters: {list(_pm.peft_config)}"
assert all(p.requires_grad == (".default." in n) for n, p in _pm.named_parameters() if "lora_" in n), "a non-policy adapter trains"
_d = {n: p for n, p in _pm.named_parameters() if ".default." in n and "lora_" in n}
assert all(torch.equal(p, _pm.get_parameter(n.replace(".default.", ".ref."))) for n, p in _d.items()), "ref != S"
assert trainer._tokenize_prompts([train_dataset[0]["prompt"]])[0][0] == chat_ids(train_dataset[0]["prompt"]), \
    "the trainer's prompt tokens differ from chat_ids (chat_template_kwargs?)"
print(f"[adapters] {sorted(_pm.peft_config)}; policy trainable ({sum(p.numel() for p in _d.values()):,} params), ref = S; "
      "trainer prompt tokens = chat_ids")

trainer.vllm_generation.sync_weights()
POLICY = "S"
_n = 0
for m in trainer.model.modules():
    if getattr(type(m).__call__, "__module__", "").startswith("vllm") and not hasattr(m, "do_not_compile"):
        m.do_not_compile = True
        _n += 1
print(f"[vLLM patch] {_n} module(s) marked do_not_compile")

def load_policy(path):
    """An adapter's weights into "default" (the policy), synced to vLLM."""
    from safetensors.torch import load_file
    from peft import set_peft_model_state_dict
    res = set_peft_model_state_dict(trainer.model, load_file(f"{path}/adapter_model.safetensors"), adapter_name="default")
    assert not res.unexpected_keys, res.unexpected_keys[:4]
    trainer.vllm_generation.sync_weights()

def trainer_logps(ids_prompt, ids_completion, temperature):
    inp = torch.tensor([ids_prompt + ids_completion], device=trainer.model.device)
    with torch.no_grad():
        logits = trainer.model(input_ids=inp).logits[0, len(ids_prompt) - 1:-1].float() / temperature
    return torch.log_softmax(logits, -1).gather(1, inp[0, len(ids_prompt):, None])[:, 0].cpu().numpy()

SAMPLER_MAX_DIFF = 0.02
_ids = [chat_ids(countdown_prompt(p["numbers"], p["target"])) for p in GRADE[4][:8]]
_outs = trainer.vllm_generation.llm.generate([{"prompt_token_ids": x} for x in _ids],
                                             SamplingParams(temperature=0.0, max_tokens=128, logprobs=1), use_tqdm=False)
_dd = []
for x, o in zip(_ids, _outs):
    t = list(o.outputs[0].token_ids)
    _dd.append(trainer_logps(x, t, 1.0) - np.array([lp[i].logprob for lp, i in zip(o.outputs[0].logprobs, t)]))
_seq = np.array([s.sum() for s in _dd])
_dd = np.concatenate(_dd)
print(f"[sampler check] {len(_dd)} tokens: mean |diff| {np.abs(_dd).mean():.4f}, signed {_dd.mean():+.4f}; "
      f"per-sequence log-ratio {np.round(_seq, 3).tolist()}")
assert np.abs(_dd).mean() < SAMPLER_MAX_DIFF and abs(_dd.mean()) < 0.01, "vLLM and the trainer disagree"

def generate_graded(puzzles_by_size, n, temperature, name):
    """The policy in the engine: n samples per puzzle (cap, seed SEED), labeled as S's grade file."""
    batch = [(s, p) for s in (4, 3) for p in puzzles_by_size[s]]
    ids = [chat_ids(countdown_prompt(p["numbers"], p["target"])) for _, p in batch]
    outs = trainer.vllm_generation.llm.generate([{"prompt_token_ids": x} for x in ids],
                                                SamplingParams(n=n, temperature=temperature, top_p=1.0,
                                                               max_tokens=MAX_COMPLETION_LENGTH, seed=SEED), use_tqdm=False)
    assert all(list(o.prompt_token_ids) == x for o, x in zip(outs, ids))
    rows = [{"model": name, "n_numbers": s, "puzzle": p.get("puzzle", i), "numbers": json.dumps(p["numbers"]),
             "target": p["target"], "sample": k, "completion": c.text, "n_tokens": len(c.token_ids),
             "finished": c.finish_reason == "stop", "truncated": c.finish_reason != "stop"}
            for i, ((s, p), o) in enumerate(zip(batch, outs)) for k, c in enumerate(o.outputs)]
    return label(pd.DataFrame(rows))

print("[build] trainer ready")

In [ ]:
# [judge] Part 1's judge into the trainer's model as a frozen adapter "judge". S's grade file at the cap (S_GRADE: N_GRADE
# puzzles per size x 8, T 0.9, as S's training rollouts; generated here if missing). Calibration (once, saved to
# CALIB_PATH), scored as the GRPO reward scores: weight = within-puzzle SD of the format reward (all completions) / of the
# judge score (scored completions); missing = the median judge score of the scored wrong completions. Self-judge check: within-puzzle win rate of correct over wrong (ties 1/2) for
# the judge, S itself (frozen "ref"), the untrained model and "shorter".
if "judge" not in trainer.model.peft_config:
    trainer.model.load_adapter(JUDGE_PATH, adapter_name="judge", is_trainable=False)
    trainer.model.set_adapter("default")
    _freeze_others(trainer.model)
assert all(p.requires_grad == (".default." in n) for n, p in trainer.model.named_parameters() if "lora_" in n)
if not os.path.exists(S_GRADE):
    load_policy(S_ADAPTER)
    os.makedirs(os.path.dirname(S_GRADE), exist_ok=True)
    generate_graded({s: v[:N_GRADE] for s, v in GRADE.items()}, GRADE_SAMPLES, TEMPERATURE, "sft").to_csv(S_GRADE, index=False)
    print(f"[judge] S graded: {S_GRADE}")
_cal_csv = CALIB_PATH.replace(".json", "_scores.csv")
if os.path.exists(CALIB_PATH):
    cal, ev = json.load(open(CALIB_PATH)), pd.read_csv(_cal_csv, keep_default_na=False)
    assert cal["judge"] == JUDGE_PATH and cal["reduction"] == JUDGE_REDUCTION, (cal["judge"], cal["reduction"])
    print(f"[judge] calibration loaded: {CALIB_PATH}")
else:
    t0 = time.time()
    ev = read_graded(S_GRADE)
    _gp = {(n, p["puzzle"]) for n, v in GRADE.items() for p in v}
    ev = ev[[(n, p) in _gp for n, p in zip(ev.n_numbers, ev.puzzle)]].reset_index(drop=True)
    texts = ev.completion.tolist()
    nums = [json.loads(x) for x in ev.numbers]
    items = [(n, int(t), judge_input(x)) for n, t, x in zip(nums, ev.target, texts)]
    ev["format_reward"] = format_total(texts)
    ev["correct_r"] = np.array(R.correctness_reward_func(None, texts, nums, ev.target.astype(int).tolist())) > 0
    ev["scored"] = [x is not None for _, _, x in items]
    for name, adapter in (("judge", "judge"), ("S", "ref"), ("base", None)):
        ev[f"{name}_mean"], ev[f"{name}_sum"] = judge_scores(trainer.model, items, adapter=adapter)
    ev["shorter"] = [-len(tok(x, add_special_tokens=False)["input_ids"]) if x else np.nan for _, _, x in items]
    sc = ev[ev.scored]
    sd_within = lambda d, col: float(np.sqrt(d.groupby(["n_numbers", "puzzle"])[col].var(ddof=0).mean()))
    key = f"judge_{JUDGE_REDUCTION}"
    cal = {"format_sd_within": sd_within(ev, "format_reward"), "judge_sd_within": sd_within(sc, key),
           "missing": float(sc.loc[~sc.correct_r, key].median() if MISSING_AT == "wrong_median" else sc[key].quantile(0.05)), "reduction": JUDGE_REDUCTION, "completions": len(ev),
           "scored": int(ev.scored.sum()), "judge": JUDGE_PATH, "source": S_GRADE}
    cal["weight"] = cal["format_sd_within"] / cal["judge_sd_within"]
    ev.drop(columns=["completion"]).to_csv(_cal_csv, index=False)
    json.dump(cal, open(CALIB_PATH, "w"), indent=1)
    print(f"[judge] calibrated on {len(ev)} completions in {(time.time() - t0) / 60:.1f} min; saved {CALIB_PATH}")
_sc = ev[ev.scored.astype(str) == "True"]
_v = _sc[f"judge_{JUDGE_REDUCTION}"].astype(float)
_want = (_v[~_sc.correct_r.astype(str).eq("True")].median() if MISSING_AT == "wrong_median" else _v.quantile(0.05))
assert abs(cal["missing"] - _want) < 1e-6, f"{CALIB_PATH}: missing {cal['missing']} is not MISSING_AT {MISSING_AT} ({_want})"
JUDGE_WEIGHT, JUDGE_MISSING = cal["weight"] * JUDGE_WEIGHT_SCALE, cal["missing"]
print(f"[judge] adapters {sorted(trainer.model.peft_config)} (only default trains); {JUDGE_REDUCTION}: weight "
      f"{JUDGE_WEIGHT:.4g} (format SD {cal['format_sd_within']:.3f} / judge SD {cal['judge_sd_within']:.3f}), missing "
      f"{JUDGE_MISSING:.1f}; scored {cal['scored']} of {cal['completions']}")
_rows = []
for size in (4, 3):
    x = ev[(ev.n_numbers == size) & (ev.scored.astype(str) == "True")]
    _k = (x.n_numbers.astype(str) + "-" + x.puzzle.astype(str)).tolist()
    _c = x.correct_r.astype(str).eq("True").values
    r = {"numbers": size, "scored": len(x), "correct": _c.mean()}
    for m in ("judge_sum", "judge_mean", "S_sum", "base_sum", "shorter"):
        r[m], r["pairs"] = within_puzzle_win_rate(_k, x[m].astype(float).values, _c)
    _rows.append(r)
with pd.option_context("display.float_format", "{:.3f}".format, "display.width", 250, "display.max_columns", None):
    print(f"\n[self-judge check] S's grade file ({N_GRADE} per size x {GRADE_SAMPLES}, T {TEMPERATURE:g}, cap {MAX_COMPLETION_LENGTH}), "
          "scored completions: within-puzzle win rate of correct over wrong (ties 1/2; 0.5 = no signal)\n"
          + pd.DataFrame(_rows).to_string(index=False))

In [ ]:
# [rft] Rejection-sampling fine-tuning rounds (Consistency_RFT_Countdown34's [rft]; settings in [config]). Per round r:
# the next RFT_PUZZLES puzzles of the training file (in order: four- and three-number puzzles alternate in blocks of 4)
# x NUM_GENERATIONS samples (vLLM, T TEMPERATURE, seed SEED + r) -> scores (format + judge, or checker) -> per puzzle
# the top RFT_KEEP finished completions above the group mean, minus those below the RFT_SKIP_Q quantile -> SFT (policy
# LoRA, loss on completion tokens) -> vLLM synced -> the held-out curve (as the GRPO runs' CurveEval). After the last
# round: GRADE_ROUNDS graded on S's grade puzzles. Saved in RUN_DIR: round<r>/ (adapter_model.safetensors +
# adapter_config.json, optimizer.pt, pool.csv: every completion with its scores, selected.csv), rft_log.csv (per SFT
# batch), rounds.csv (per round), curve_n34x{CURVE_N}.csv (round 0 = S), grade/. A rerun resumes after the last round.
import random
from datetime import datetime
from safetensors.torch import load_file, save_file
from peft import get_peft_model_state_dict, set_peft_model_state_dict
sys.path.insert(0, f"{DRIVE}/code")
from runs import log_run                        # Consistency/runs.csv: one row per run start

policy = trainer.model
_STOP_IDS = {tok.eos_token_id, EOT}
assert len(train_dataset) >= RFT_ROUNDS * RFT_PUZZLES, (len(train_dataset), RFT_ROUNDS * RFT_PUZZLES)
_named = [(n, p) for n, p in policy.named_parameters() if p.requires_grad]
assert _named and all(".default." in n and "lora_" in n for n, _ in _named), "only the policy LoRA may be trainable"
_params = [p for _, p in _named]
opt = torch.optim.AdamW(_params, lr=RFT_LR, betas=(0.9, ADAM_BETA2), weight_decay=WEIGHT_DECAY)
SELECT_WEIGHT = JUDGE_WEIGHT if SELECT == "judge" else 0.0     # the judge's weight in the selection score

def curve_eval(step, out_prefix):
    """The GRPO notebook's held-out curve: CURVE greedy x 1 and sampled x CURVE_K."""
    t0 = time.time()
    rows = {}
    for kind, n, T in (("greedy", 1, 0.0), ("sampled", CURVE_K, TEMPERATURE)):
        d = generate_graded(CURVE, n, T, f"step{step}")
        d.to_csv(f"{out_prefix}_{kind}.csv", index=False)
        for s, x in d.groupby("n_numbers"):
            ok = x.finished & x.correct
            rows[f"{kind} acc {s}"] = ok.mean()
            if kind == "sampled":
                rows[f"pass@{CURVE_K} {s}"] = x.assign(ok=ok).groupby("puzzle").ok.any().mean()
                rows[f"truncated {s}"] = 1 - x.finished.mean()
                rows[f"correct among answered {s}"] = ok.sum() / max((x.finished & x.answered).sum(), 1)
    return {"step": step, **rows, "minutes": (time.time() - t0) / 60}

SAFE_TO_CHANGE = {"GRADE_ROUNDS", "VLLM_GPU_MEMORY_UTILIZATION", "UNASSIGN_WHEN_DONE", "COMPARE_RUNS", "MICRO_BATCH",
                  "SAVE_STEPS", "DUMP_EVERY"}
_fn = lambda x: json.loads(re.sub(r"<function (\w+) at 0x[0-9a-f]+>", r"\1", json.dumps(x, default=lambda o: getattr(o, "__name__", str(o)))))

def run_config():
    """The settings saved in run.json: upper-case globals, functions by name, minus runtime state."""
    return _fn({k: v for k, v in globals().items() if k.isupper() and not k.startswith("_")
                and k not in ("GRADE", "CURVE", "SAFE_TO_CHANGE", "CUR", "POLICY")
                and isinstance(v, (int, float, str, bool, tuple, list, dict, type(None)))})

def check_config(old):
    """A rerun of the folder: its saved settings vs these. Safe keys are printed; any other difference asks."""
    new = run_config()
    oc = {k: v for k, v in _fn(old.get("config", {})).items() if not k.startswith("_") and k not in ("CUR", "POLICY")}
    diff = {k: (oc.get(k, "(absent)"), new.get(k, "(absent)")) for k in set(oc) | set(new) if oc.get(k) != new.get(k)}
    if old.get("calibration") != cal:
        diff["calibration"] = (old.get("calibration"), cal)
    ask = {k: v for k, v in diff.items() if k not in SAFE_TO_CHANGE}
    for k, (a, b) in sorted(diff.items()):
        print(f"[rft] setting {'CHANGED' if k in ask else 'changed'}: {k}: {str(a)[:150]} -> {str(b)[:150]}")
    if ask and input(f"[rft] continue {RUN_ID} with the settings above? [y/N] ").strip().lower() != "y":
        raise RuntimeError(f"[rft] not continued; change RUN_TAG for a new run or restore the settings")
    return bool(diff)

def round_dir(r):
    return f"{RUN_DIR}/round{r}"

def save_round(r):
    d = round_dir(r)
    os.makedirs(d, exist_ok=True)
    save_file({k: v.detach().cpu().contiguous() for k, v in get_peft_model_state_dict(policy, adapter_name="default").items()},
              f"{d}/adapter_model.safetensors")
    shutil.copy(f"{S_ADAPTER}/adapter_config.json", f"{d}/adapter_config.json")
    torch.save(opt.state_dict(), f"{d}/optimizer.pt")           # last: its presence marks the round as done

def load_round(r, optimizer=True):
    load_policy(round_dir(r))
    if optimizer:
        opt.load_state_dict(torch.load(f"{round_dir(r)}/optimizer.pt"))

def score_pool(ds, outs):
    """Every completion of the round with its scores: format, checker (measured; selected on only with SELECT
    'checker'), the judge's raw score and its reward value (missing = JUDGE_MISSING), and the selection score."""
    rows = []
    for i, o in enumerate(outs):
        for k, c in enumerate(o.outputs):
            rows.append({"pid": ds[i]["pid"], "n_numbers": ds[i]["n"], "puzzle": i,
                         "numbers": json.dumps([int(x) for x in ds[i]["numbers"]]), "target": int(ds[i]["target"]),
                         "sample": k, "completion": c.text, "n_tokens": len(c.token_ids),
                         "finished": c.finish_reason == "stop", "_ids": list(c.token_ids)})
    df = pd.DataFrame(rows)
    texts, nums = df.completion.tolist(), [json.loads(x) for x in df.numbers]
    if CHAT_KWARGS.get("enable_thinking") is False:
        assert not any("<think>" in t for t in texts), "a completion opens <think>: thinking is not off"
    df["format"] = format_total(texts)
    df["checker_reward"] = np.array(R.correctness_reward_func(None, texts, nums, df.target.tolist()), dtype=float)
    df["correct"] = df.checker_reward > 0
    m, s = judge_scores(policy, [(n, int(t), judge_input(x)) for n, t, x in zip(nums, df.target, texts)])
    raw = m if JUDGE_REDUCTION == "mean" else s
    df["judge_scored"], df[f"judge_{JUDGE_REDUCTION}"] = ~np.isnan(raw), raw
    df["judge_value"] = np.where(np.isnan(raw), JUDGE_MISSING, raw)
    df["score"] = df.format + (SELECT_WEIGHT * df.judge_value if SELECT == "judge" else df.checker_reward)
    return df

def select(df):
    """Per puzzle: finished completions above the group mean score, best RFT_KEEP of them (ties: lower sample index);
    then, with RFT_SKIP_Q, only those scored at or above that quantile of the round's selected scores."""
    g = df.groupby("puzzle").score.transform("mean")
    cand = df[df.finished & (df.score > g + 1e-9)]
    sel = cand.sort_values(["puzzle", "score", "sample"], ascending=[True, False, True]).groupby("puzzle").head(RFT_KEEP)
    if RFT_SKIP_Q and len(sel):
        sel = sel[sel.score >= sel.score.quantile(RFT_SKIP_Q)]
    return sel

def sft(ds, sel, r):
    """One SFT pass per epoch over the selected completions; returns the batch log rows."""
    seqs = []
    for _, row in sel.iterrows():
        c = list(row["_ids"])
        if c and c[-1] not in _STOP_IDS:              # finished by a stop token vLLM left out: teach the stop
            c = c + [EOT]
        seqs.append((chat_ids(ds[int(row["puzzle"])]["prompt"]), c))
    policy.config.use_cache = False
    policy.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    policy.train()
    log, rng = [], random.Random(SEED + r)
    for ep in range(RFT_EPOCHS):
        rng.shuffle(seqs)
        for b in range(0, len(seqs), RFT_BATCH):
            batch = seqs[b:b + RFT_BATCH]
            n_tok = sum(len(c) for _, c in batch)
            opt.zero_grad(set_to_none=True)
            tot = 0.0
            for mb in range(0, len(batch), RFT_MICRO):
                chunk = batch[mb:mb + RFT_MICRO]
                L = max(len(p) + len(c) for p, c in chunk)
                ids = torch.full((len(chunk), L), tok.pad_token_id, dtype=torch.long)
                attn, mask = torch.zeros_like(ids), torch.zeros((len(chunk), L - 1), dtype=torch.float32)
                for j, (p, c) in enumerate(chunk):
                    ids[j, :len(p) + len(c)], attn[j, :len(p) + len(c)] = torch.tensor(p + c), 1
                    mask[j, len(p) - 1:len(p) + len(c) - 1] = 1.0          # positions that predict completion tokens
                ids, attn, mask = ids.to(policy.device), attn.to(policy.device), mask.to(policy.device)
                with torch.autocast("cuda", dtype=torch.bfloat16):
                    logits = policy(input_ids=ids, attention_mask=attn).logits[:, :-1]
                lp = selective_log_softmax(logits, ids[:, 1:]).float()
                loss = -(lp * mask).sum() / n_tok
                loss.backward()
                tot += float(loss)
                del logits, lp, loss
            gn = float(torch.nn.utils.clip_grad_norm_(_params, RFT_MAX_GRAD_NORM))
            opt.step()
            log.append({"round": r, "epoch": ep, "batch": b // RFT_BATCH, "nll_per_token": tot, "grad_norm": gn,
                        "sequences": len(batch), "tokens": n_tok})
    policy.eval()
    opt.zero_grad(set_to_none=True)
    return log

def grade_path(r):
    return f"{RUN_DIR}/grade/gen_{RUN_ID}-round{r}.csv"

# ---- run folder, resume
os.makedirs(RUN_DIR, exist_ok=True)
_old = json.load(open(f"{RUN_DIR}/run.json")) if os.path.exists(f"{RUN_DIR}/run.json") else None
_changed = check_config(_old) if _old is not None else False
_done = [r for r in range(1, RFT_ROUNDS + 1) if os.path.exists(f"{round_dir(r)}/optimizer.pt")]
_last = max(_done, default=0)
assert _done == list(range(1, _last + 1)), f"rounds on Drive are not contiguous: {_done}"
if _last:
    load_round(_last)
    print(f"[rft] resuming after round {_last}: policy and optimizer loaded from {round_dir(_last)}")
else:
    load_policy(S_ADAPTER)
    _want = load_file(f"{S_ADAPTER}/adapter_model.safetensors")
    _now = get_peft_model_state_dict(policy, adapter_name="default")
    assert max((_now[k].detach().float().cpu() - _want[k].float()).abs().max().item() for k in _want) == 0, "policy != S"
_hist = (_old.get("history", []) + [{k: _old.get(k) for k in ("created", "finished", "config", "calibration")}]
         if _changed else (_old or {}).get("history", []))
json.dump({"id": RUN_ID, "kind": "rft", "notebook": "Consistency_RFT_Qwen_Countdown",
           "parents": {"model": MODEL, "start": S_ADAPTER, "judge": JUDGE_PATH if SELECT == "judge" else None},
           "created": (_old or {}).get("created", datetime.now().isoformat()), "calibration": cal,
           "config": run_config(), "history": _hist,
           "status": "trained" if _last == RFT_ROUNDS else "training",
           **({"finished": _old["finished"]} if _old and _old.get("finished") and _last == RFT_ROUNDS else {})},
          open(f"{RUN_DIR}/run.json", "w"), indent=1, default=str)
if _last < RFT_ROUNDS:
    log_run("Consistency_RFT_Qwen_Countdown",
            f"RFT ({SELECT} selection), SFT start, {RFT_ROUNDS} rounds x {RFT_PUZZLES} x {NUM_GENERATIONS}, cap "
            f"{MAX_COMPLETION_LENGTH}" + (f", resumed after round {_last}" if _last else ""), MODEL, RUN_DIR)
print(f"[rft] {RUN_ID}: selection {SELECT}" + (f" (judge weight {SELECT_WEIGHT:.4g}, missing {JUDGE_MISSING:.1f})"
      if SELECT == "judge" else "") + f"; rounds done {_last} of {RFT_ROUNDS}")

# ---- held-out curve: round 0 = S (cached in S_DIR, shared with the GRPO runs)
if not os.path.exists(f"{CURVE_S}_row.json"):
    load_policy(S_ADAPTER)
    os.makedirs(os.path.dirname(CURVE_S), exist_ok=True)
    json.dump(curve_eval(0, CURVE_S), open(f"{CURVE_S}_row.json", "w"), indent=1)
    if _last:
        load_round(_last, optimizer=False)
_curve_csv = f"{RUN_DIR}/curve_n34x{CURVE_N}.csv"
if not os.path.exists(_curve_csv):
    pd.DataFrame([{"round": 0, "rollouts": 0, **json.load(open(f"{CURVE_S}_row.json"))}]).to_csv(_curve_csv, index=False)

def curve_round(r):
    """The held-out curve row of round r (the policy must be round r's, synced), unless it exists."""
    rows = pd.read_csv(_curve_csv).to_dict("records")
    if any(int(x["round"]) == r for x in rows):
        return
    os.makedirs(f"{RUN_DIR}/curve", exist_ok=True)
    rows.append({"round": r, "rollouts": r * RFT_PUZZLES * NUM_GENERATIONS, **curve_eval(r, f"{RUN_DIR}/curve/round{r:02d}")})
    pd.DataFrame(rows).to_csv(_curve_csv, index=False)

_rounds_csv, _log_csv = f"{RUN_DIR}/rounds.csv", f"{RUN_DIR}/rft_log.csv"
for r in range(_last + 1, RFT_ROUNDS + 1):
    t0 = time.time()
    ds = train_dataset.select(range((r - 1) * RFT_PUZZLES, r * RFT_PUZZLES))
    ids = [chat_ids(p) for p in ds["prompt"]]
    trainer.vllm_generation.sync_weights()
    outs = trainer.vllm_generation.llm.generate(
        [{"prompt_token_ids": x} for x in ids],
        SamplingParams(n=NUM_GENERATIONS, temperature=TEMPERATURE, max_tokens=MAX_COMPLETION_LENGTH, seed=SEED + r),
        use_tqdm=True)
    assert all(list(o.prompt_token_ids) == x for o, x in zip(outs, ids)), "vLLM returned outputs for other prompts"
    t_gen = time.time()
    pool = score_pool(ds, outs)
    sel = select(pool)
    t_score = time.time()
    log = sft(ds, sel, r)
    t_sft = time.time()
    save_round(r)
    pool.drop(columns="_ids").to_csv(f"{round_dir(r)}/pool.csv", index=False)
    sel.drop(columns="_ids").to_csv(f"{round_dir(r)}/selected.csv", index=False)
    pd.concat(([pd.read_csv(_log_csv)] if os.path.exists(_log_csv) else []) + [pd.DataFrame(log)]).to_csv(_log_csv, index=False)

    trainer.vllm_generation.sync_weights()
    curve_round(r)
    cv = pd.read_csv(_curve_csv).set_index("round").loc[r]
    row = {"round": r, "rollouts": r * RFT_PUZZLES * NUM_GENERATIONS,
           "pool correct": pool.correct.mean(), "pool finished": pool.finished.mean(),
           "pool correct 4": pool.correct[pool.n_numbers == 4].mean(), "pool correct 3": pool.correct[pool.n_numbers == 3].mean(),
           "puzzles with a selection": sel.puzzle.nunique() / RFT_PUZZLES, "selected": len(sel),
           "selected correct (measured)": sel.correct.mean() if len(sel) else float("nan"),
           "selected tokens": sel.n_tokens.mean() if len(sel) else float("nan"),
           "sft nll/token (last batch)": log[-1]["nll_per_token"] if log else float("nan"),
           **{f"curve {k}": cv[k] for k in ("greedy acc 4", "greedy acc 3", "sampled acc 4", "sampled acc 3")},
           "min generate": (t_gen - t0) / 60, "min score": (t_score - t_gen) / 60, "min sft": (t_sft - t_score) / 60,
           "min total": (time.time() - t0) / 60}
    pd.concat(([pd.read_csv(_rounds_csv)] if os.path.exists(_rounds_csv) else []) + [pd.DataFrame([row])]).to_csv(
        _rounds_csv, index=False)
    with pd.option_context("display.float_format", "{:.3f}".format, "display.width", 250, "display.max_columns", None):
        print(f"\n[rft] rounds so far (pool: {RFT_PUZZLES} training puzzles x {NUM_GENERATIONS} at T {TEMPERATURE:g}; "
              f"curve: held-out {CURVE_N} per size, greedy x 1 and sampled x {CURVE_K} at T {TEMPERATURE:g})\n"
              + pd.read_csv(_rounds_csv).to_string(index=False))
if _last < RFT_ROUNDS:
    _r = json.load(open(f"{RUN_DIR}/run.json"))
    _r.update(status="trained", finished=datetime.now().isoformat())
    json.dump(_r, open(f"{RUN_DIR}/run.json", "w"), indent=1, default=str)

# ---- catch-up (a round saved before its curve row) and the grades of GRADE_ROUNDS, each with that round's adapter
for r in sorted(set(range(1, RFT_ROUNDS + 1)) - set(pd.read_csv(_curve_csv)["round"].astype(int))):
    load_round(r, optimizer=False)
    curve_round(r)
for r in GRADE_ROUNDS:
    if not os.path.exists(grade_path(r)):
        load_round(r, optimizer=False)
        os.makedirs(os.path.dirname(grade_path(r)), exist_ok=True)
        generate_graded({s: v[:N_GRADE] for s, v in GRADE.items()}, GRADE_SAMPLES, TEMPERATURE,
                        os.path.basename(grade_path(r))[4:-4]).to_csv(grade_path(r), index=False)
        print(f"[grade] saved {grade_path(r)}")
with pd.option_context("display.float_format", "{:.3f}".format, "display.width", 250, "display.max_columns", None):
    print(f"\n[curve] held-out {CURVE_N} per size: greedy x 1, sampled x {CURVE_K} at T {TEMPERATURE:g} (round 0 = S)\n"
          + pd.read_csv(_curve_csv).to_string(index=False))
print(f"[rft] done: {RFT_ROUNDS} rounds in {RUN_DIR}")

In [ ]:
# [compare] S's grade file (S_GRADE: same puzzles, 8 x T 0.9, at the cap, seed 0), the RFT rounds in GRADE_ROUNDS and
# the graded GRPO runs in COMPARE_RUNS (final adapters, 150 steps = 19,200 rollouts = RFT round 4): the table per size,
# then per-puzzle accuracy differences with 95% bootstrap CIs over puzzles.
S_g = read_graded(S_GRADE)
G = {"S": S_g}
for r in GRADE_ROUNDS:
    G[f"RFT round {r} ({r * RFT_PUZZLES * NUM_GENERATIONS:,} rollouts)"] = read_graded(grade_path(r))
for name, rid in COMPARE_RUNS.items():
    p = f"{DRIVE}/countdown/grpo/{rid}/grade/gen_{rid}.csv"
    if os.path.exists(p):
        d = read_graded(p)
        assert len(d) == len(S_g) and set(zip(d.n_numbers, d.puzzle)) == set(zip(S_g.n_numbers, S_g.puzzle)), p
        G[f"{name} (150 steps, 19,200 rollouts)"] = d
    else:
        print(f"[compare] {name}: no grade file {p}")
tab = pd.DataFrame([{"policy": a, "numbers": s, **stats(x)} for a, d in G.items() for s, x in d.groupby("n_numbers")]
                   ).sort_values(["numbers"], kind="stable")
tab.to_csv(f"{RUN_DIR}/compare.csv", index=False)
with pd.option_context("display.float_format", "{:.3f}".format, "display.width", 250, "display.max_columns", None):
    print(f"[compare] S's grade puzzles, {N_GRADE} x {GRADE_SAMPLES} per size, T {TEMPERATURE:g}, cap {MAX_COMPLETION_LENGTH}\n"
          + tab.to_string(index=False))
_rng = np.random.default_rng(SEED)

def diff_ci(a, b):
    """Per-puzzle accuracy a - b (same puzzles), mean and 95% bootstrap."""
    acc = lambda d: d.assign(ok=d.finished & d.correct).groupby(["n_numbers", "puzzle"]).ok.mean()
    out = []
    for s in (4, 3):
        dd = (acc(a[a.n_numbers == s]) - acc(b[b.n_numbers == s])).dropna().values
        bs = dd[_rng.integers(0, len(dd), (10000, len(dd)))].mean(1)
        out.append(f"{s}-number {dd.mean():+.3f} [{np.percentile(bs, 2.5):+.3f}, {np.percentile(bs, 97.5):+.3f}]")
    return "; ".join(out)

_names = list(G)
for a in _names[1:]:
    print(f"{a} - S: {diff_ci(G[a], S_g)}")
for a in [x for x in _names if x.startswith("RFT")]:
    for b in [x for x in _names if x.startswith("GRPO")]:
        print(f"{a} - {b}: {diff_ci(G[a], G[b])}")

In [ ]:
# [release] All results are on Drive: release the runtime if UNASSIGN_WHEN_DONE (a failed cell stops Run all before this).
if UNASSIGN_WHEN_DONE:
    from google.colab import runtime
    runtime.unassign()